# R01 — Supervised ResNet50 (full eligible training set)
**Kaggle notebook name for the first run:** `lesion-r01-supervised-resnet50-s42`

**Attach completed notebook outputs:** `lesion-n01-dataset-audit-grouped-splits` (images and N01 manifest) and `lesion-r00-protocol-lock` (locked protocol and development splits). **GPU required.** Enable Internet for the first ImageNet weight download, or attach a verified torchvision ResNet50 IMAGENET1K_V1 state dict and set `IMAGENET_WEIGHTS_PATH`.

One seed per Kaggle execution (default `SEED=42`). Repeat with 43–46 under names `lesion-r01-supervised-resnet50-s43` … `-s46`; each run exports its own files. To resume a seed, attach that seed's preceding completed output or set `R01_RESUME_CHECKPOINT`. **Do not attach/test against the test split in R01.**

Method: ImageNet-initialized PyTorch ResNet50, eight-class head, 8 frozen-head epochs, up to 35 fine-tuning epochs, frozen BatchNorm, historical sqrt-inverse-frequency class weighting, best checkpoint by validation balanced accuracy. Differences from the historical TensorFlow/Keras backend are recorded explicitly. All 16,490 training images are used. The 2,267 meta-training images are used for inference only, not for base-model fitting or validation selection.

In [1]:
import os,json,random,time,hashlib,shutil,platform,sys,gc
from pathlib import Path
from datetime import datetime,timezone
import numpy as np,pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset,DataLoader
from torchvision import transforms
from torchvision.models import resnet50,ResNet50_Weights
from sklearn.metrics import accuracy_score,balanced_accuracy_score,f1_score,confusion_matrix,classification_report
import matplotlib.pyplot as plt

SEED=42
SEEDS=[42,43,44,45,46]
HEAD_EPOCHS=8
FINETUNE_EPOCHS=35
HEAD_LR=1e-3
FINETUNE_LR=3e-5
BATCH_SIZE=32
EVAL_BATCH=64
WORKERS=min(4,os.cpu_count() or 2)
PATIENCE=8
WEIGHT_DECAY=1e-4
DROPOUT=0.30
AMP=True
TIME_LIMIT_HOURS=8.5
IMAGENET_WEIGHTS_PATH=os.getenv('R01_IMAGENET_WEIGHTS','').strip()
INPUT_DIR=Path(os.getenv('R01_INPUT_DIR','/kaggle/input'))
OUT=Path(os.getenv('R01_OUTPUT_DIR',f'/kaggle/working/lesion-r01-supervised-resnet50-s{SEED}' if Path('/kaggle/working').exists() else f'./lesion-r01-supervised-resnet50-s{SEED}')).resolve()
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type!='cuda' and os.getenv('R01_ALLOW_CPU','0')!='1':raise RuntimeError('Enable Kaggle GPU. CPU runs are intended only for local tests.')
if SEED not in SEEDS:raise ValueError('SEED must be one of the five R00 seeds')
for d in ['models','training_state','predictions','reports','figures','tables']:(OUT/d).mkdir(parents=True,exist_ok=True)
START=time.monotonic()
CLASSES=['MEL','NV','BCC','AK','BKL','DF','VASC','SCC']

os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False
    torch.use_deterministic_algorithms(True,warn_only=True)

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(2**20),b''):h.update(block)
    return h.hexdigest()

def dump(path,obj):Path(path).write_text(json.dumps(obj,indent=2,allow_nan=False)+'\n',encoding='utf-8')
def require(ok,msg):
    if not bool(ok):raise RuntimeError(msg)

seed_all(SEED)
print('GPU:',torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else 'local CPU test','torch:',torch.__version__)
print('Output:',OUT)

GPU: Tesla T4 torch: 2.11.0+cu130
Output: /kaggle/working/lesion-r01-supervised-resnet50-s42


## 1. Load N01 images through the executed R00 protocol

In [2]:
def one_root(marker,field,value,override=''):
    if override:
        roots=[Path(override).expanduser().resolve()]
    else:
        roots=sorted({p.parent.parent for p in INPUT_DIR.rglob(marker)})
    valid=[]
    for root in roots:
        path=root/marker
        if not path.is_file():continue
        try:meta=json.loads(path.read_text())
        except (ValueError,OSError):continue
        if meta.get(field)==value:valid.append(root)
    require(len(valid)==1,f'Expected exactly one {value} input with {marker}; found {valid}')
    return valid[0]

R00=one_root('configs/experiment_protocol.json','name','lesion-r00-protocol-lock',os.getenv('R01_R00_ROOT',''))
N01=one_root('reports/protocol.json','project','lesion-n01-dataset-audit-grouped-splits',os.getenv('R01_N01_ROOT',''))
protocol=json.loads((R00/'configs/experiment_protocol.json').read_text())
n01_meta=json.loads((N01/'reports/protocol.json').read_text())
checks=json.loads((R00/'reports/artifact_checksums.json').read_text())
for rel in ['configs/experiment_protocol.json','splits/train.csv','splits/validation.csv']:
    require(sha256(R00/rel)==checks.get(rel),f'R00 artifact checksum mismatch: {rel}')
require(sha256(N01/'splits/experiment_master_manifest.csv')==protocol['n01_manifest_sha256'],'Wrong N01 manifest')
require(n01_meta['manifest_sha256']==protocol['n01_manifest_sha256'],'N01/R00 incompatible')
require(protocol['status']=='LOCKED' and protocol['classes']==CLASSES,'Wrong R00 protocol or class ordering')
require(protocol['use_reduced_label_budgets'] is False,'Reduced-label protocol is forbidden')
require(protocol['random_seeds']==SEEDS,'Seed protocol differs from R00')
require(protocol['data_access']['supervised_fit']==['train'],'Invalid supervised-fit permission')

frames={}
for split in ['train','meta_train','validation']:
    df=pd.read_csv(R00/'splits'/f'{split}.csv',dtype={'image_id':str,'group_id':str,'lesion_id':str,'source_sha256':str,'image_relpath':str,'label':str})
    require(len(df)==protocol['split_counts'][split],f'Wrong {split} count')
    require(df.split.eq(split).all() and df.image_id.is_unique,f'{split} IDs invalid')
    require(df.label.isin(CLASSES).all(),'Unexpected class')
    require((df.class_index.to_numpy(dtype=int)==df.label.map({c:i for i,c in enumerate(CLASSES)}).to_numpy(dtype=int)).all(),'Class indexes disagree')
    require(df.group_id.notna().all() and df.group_id.astype(str).ne('').all(),f'Missing group in {split}')
    for rel in df.image_relpath:
        p=Path(rel)
        require(not p.is_absolute() and '..' not in p.parts and p.parts[0]=='images',f'Unsafe image path: {rel}')
        require((N01/p).is_file(),f'Missing N01 image: {rel}')
    frames[split]=df.sort_values('image_id').reset_index(drop=True)
for a,b in [('train','validation'),('train','meta_train'),('meta_train','validation')]:
    require(set(frames[a].group_id).isdisjoint(frames[b].group_id),f'{a}/{b} identity overlap')
    require(set(frames[a].image_id).isdisjoint(frames[b].image_id),f'{a}/{b} image overlap')
    require(set(frames[a].source_sha256).isdisjoint(frames[b].source_sha256),f'{a}/{b} byte overlap')
require(len(frames['train'])==16490 and len(frames['meta_train'])==2267 and len(frames['validation'])==2186,'Unexpected N01 partition sizes')
master=pd.read_csv(N01/'splits/experiment_master_manifest.csv',dtype=str,keep_default_na=False)
ref=master.loc[master.split.eq('meta_train'),['image_id','class_index','group_id','image_relpath']].sort_values('image_id').reset_index(drop=True)
observed=frames['meta_train'][['image_id','class_index','group_id','image_relpath']].astype(str)
require(ref.astype(str).equals(observed),'R00 meta-training CSV does not match the SHA256-locked N01 manifest')
print('N01:',N01,'R00:',R00)
print('Development partitions:',{k:len(v) for k,v in frames.items()})
print('No test data loaded.')

N01: /kaggle/input/notebooks/rinkyranisaha/lesion-n01/lesion-n01-dataset-audit-grouped-splits R00: /kaggle/input/notebooks/sudiptoroy007/lesion-r00-protocol-lock/lesion-r00-protocol-lock
Development partitions: {'train': 16490, 'meta_train': 2267, 'validation': 2186}
No test data loaded.


## 2. Dataset, historical training recipe, and model

In [3]:
class RightAngle:
    def __call__(self,img):
        k=random.randrange(4)
        return img.rotate(k*90) if k else img

TRAIN_T=transforms.Compose([transforms.RandomHorizontalFlip(),transforms.RandomVerticalFlip(),RightAngle(),transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
EVAL_T=transforms.Compose([transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])

class Images(Dataset):
    def __init__(self,frame,augment=False):
        self.df=frame
        self.paths=[N01/Path(p) for p in frame.image_relpath]
        self.y=frame.class_index.to_numpy(dtype=np.int64)
        self.t=TRAIN_T if augment else EVAL_T
    def __len__(self):return len(self.y)
    def __getitem__(self,i):
        with Image.open(self.paths[i]) as im:x=self.t(im.convert('RGB'))
        return x,int(self.y[i])

def worker_seed(worker_id):
    seed=torch.initial_seed()%2**32
    random.seed(seed);np.random.seed(seed)

loader_args={'num_workers':WORKERS,'pin_memory':DEVICE.type=='cuda'}
if WORKERS:loader_args['prefetch_factor']=2
G=torch.Generator()
train_loader=DataLoader(Images(frames['train'],True),batch_size=BATCH_SIZE,shuffle=True,generator=G,worker_init_fn=worker_seed,**loader_args)
val_loader=DataLoader(Images(frames['validation']),batch_size=EVAL_BATCH,shuffle=False,**loader_args)
meta_loader=DataLoader(Images(frames['meta_train']),batch_size=EVAL_BATCH,shuffle=False,**loader_args)
counts=np.bincount(frames['train'].class_index.to_numpy(dtype=int),minlength=len(CLASSES))
require((counts>0).all(),'Missing training class')
raw=np.sqrt(len(frames['train'])/counts)
weights=.99*raw/raw.max()
CLASS_WEIGHTS=torch.tensor(weights,dtype=torch.float32,device=DEVICE)
pd.DataFrame({'class_index':range(8),'class_name':CLASSES,'train_count':counts,'loss_weight':weights}).to_csv(OUT/'tables/class_weights.csv',index=False)

class SupervisedResNet50(nn.Module):
    def __init__(self,weights=None):
        super().__init__()
        self.encoder=resnet50(weights=weights)
        self.encoder.fc=nn.Identity()
        self.dropout=nn.Dropout(DROPOUT)
        self.classifier=nn.Linear(2048,len(CLASSES))
    def forward(self,x):return self.classifier(self.dropout(self.encoder(x)))

def make_model(load_pretrained=True):
    if not load_pretrained:return SupervisedResNet50().to(DEVICE)
    if IMAGENET_WEIGHTS_PATH:
        model=SupervisedResNet50()
        state=torch.load(IMAGENET_WEIGHTS_PATH,map_location='cpu',weights_only=True)
        if isinstance(state,dict) and 'state_dict' in state:state=state['state_dict']
        backbone=resnet50(weights=None)
        backbone.load_state_dict(state,strict=True)
        model.encoder=backbone
        model.encoder.fc=nn.Identity()
    else:
        try:model=SupervisedResNet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        except Exception as e:raise RuntimeError('ImageNet weights unavailable. Enable Kaggle Internet or set R01_IMAGENET_WEIGHTS to the torchvision ResNet50 IMAGENET1K_V1 .pth file.') from e
    return model.to(DEVICE)

def set_phase(model,phase):
    for p in model.encoder.parameters():p.requires_grad=(phase=='finetune')
    for m in model.encoder.modules():
        if isinstance(m,nn.modules.batchnorm._BatchNorm):
            for p in m.parameters():p.requires_grad=False
    for p in model.classifier.parameters():p.requires_grad=True

def mode_train(model,phase):
    model.train()
    if phase=='head':model.encoder.eval()
    else:
        for m in model.encoder.modules():
            if isinstance(m,nn.modules.batchnorm._BatchNorm):m.eval()

print('Training images:',len(train_loader.dataset),'validation:',len(val_loader.dataset),'meta inference:',len(meta_loader.dataset))

Training images: 16490 validation: 2186 meta inference: 2267


## 3. Reproducible checkpointing and evaluation

In [4]:
CFG={'stage':'R01','name':'lesion-r01-supervised-resnet50','seed':SEED,'n01_manifest_sha256':protocol['n01_manifest_sha256'],
     'r00_protocol_sha256':checks['configs/experiment_protocol.json'],'classes':CLASSES,
     'model':'torchvision.models.resnet50','initialization':'ResNet50_Weights.IMAGENET1K_V1','image_size':224,
     'batch_size':BATCH_SIZE,'eval_batch':EVAL_BATCH,'head_epochs':HEAD_EPOCHS,'finetune_epochs':FINETUNE_EPOCHS,
     'head_lr':HEAD_LR,'finetune_lr':FINETUNE_LR,'weight_decay':WEIGHT_DECAY,'dropout':DROPOUT,
     'patience':PATIENCE,'weighting':'0.99*sqrt(N/train_class_count)/max; per-sample weighted CE averaged over batch',
     'augmentation':'H flip, V flip, uniform rotation {0,90,180,270}; ImageNet mean/std',
     'batchnorm':'frozen in fine-tuning','selection':'validation balanced_accuracy then macro_f1 then accuracy',
     'validation_scheduler':'ReduceLROnPlateau(val_unweighted_nll,factor=0.5,patience=3,min_lr=1e-7)',
     'historical_port_note':'PyTorch torchvision ImageNet1K V1 initialization/normalization; not numerically identical to Keras ResNet50 preprocessing/weights'}
SIGNATURE=hashlib.sha256(json.dumps(CFG,sort_keys=True).encode()).hexdigest()
dump(OUT/'reports/config.json',{**CFG,'config_sha256':SIGNATURE,'imagenet_weights_path':IMAGENET_WEIGHTS_PATH or None})
BEST=OUT/'models/best.pt';LAST=OUT/'training_state/last.pt';HISTORY=OUT/'tables/training_history.csv'

def score(y,prob):
    pred=prob.argmax(1)
    return {'accuracy':float(accuracy_score(y,pred)),
            'balanced_accuracy':float(balanced_accuracy_score(y,pred)),
            'macro_f1':float(f1_score(y,pred,average='macro',zero_division=0))}

def evaluate(model,loader,keep=False):
    model.eval();loss_sum=0.;n=0;ys=[];logits=[]
    with torch.inference_mode():
        for x,y in loader:
            x=x.to(DEVICE,non_blocking=True)
            with torch.autocast(device_type=DEVICE.type,enabled=AMP and DEVICE.type=='cuda'):
                z=model(x)
            loss_sum+=float(F.cross_entropy(z.float(),y.to(DEVICE),reduction='sum').item())
            n+=len(y);ys.append(y.numpy());logits.append(z.float().cpu().numpy())
    y=np.concatenate(ys);z=np.concatenate(logits)
    proba=torch.softmax(torch.from_numpy(z),dim=1).numpy()
    metrics=score(y,proba);metrics['nll']=float(loss_sum/n)
    return (metrics,y,z,proba) if keep else metrics

def snapshot(model,phase,next_epoch,optimizer=None,scheduler=None,scaler=None,bad_epochs=0,best_key=None):
    return {'signature':SIGNATURE,'seed':SEED,'phase':phase,'next_epoch':next_epoch,'bad_epochs':bad_epochs,'best_key':best_key,
            'model':model.state_dict(),'optimizer':optimizer.state_dict() if optimizer else None,
            'scheduler':scheduler.state_dict() if scheduler else None,'scaler':scaler.state_dict() if scaler else None,
            'python_rng':random.getstate(),'numpy_rng':np.random.get_state(),
            'torch_rng':torch.get_rng_state(),'cuda_rng':torch.cuda.get_rng_state_all() if DEVICE.type=='cuda' else []}

def save_last(model,phase,next_epoch,optimizer=None,scheduler=None,scaler=None,bad_epochs=0,best_key=None):
    tmp=LAST.with_suffix('.tmp');torch.save(snapshot(model,phase,next_epoch,optimizer,scheduler,scaler,bad_epochs,best_key),tmp);tmp.replace(LAST)

def restore_rng(state):
    random.setstate(state['python_rng']);np.random.set_state(state['numpy_rng']);torch.set_rng_state(state['torch_rng'])
    if DEVICE.type=='cuda' and state['cuda_rng']:torch.cuda.set_rng_state_all(state['cuda_rng'])

def locate_resume():
    explicit=os.getenv('R01_RESUME_CHECKPOINT','').strip()
    if explicit:return Path(explicit).expanduser().resolve()
    candidates=[p for p in INPUT_DIR.rglob('training_state/last.pt') if f'lesion-r01-supervised-resnet50-s{SEED}' in str(p)] if INPUT_DIR.is_dir() else []
    require(len(candidates)<=1,f'Multiple resume checkpoints detected: {candidates}')
    return candidates[0] if candidates else None

resume_path=locate_resume();state=None
if resume_path:
    state=torch.load(resume_path,map_location='cpu',weights_only=False)
    require(state.get('signature')==SIGNATURE and state.get('seed')==SEED,'Previous checkpoint config/seed differs; refusing unsafe resume')
    prev=resume_path.parent.parent
    if (prev/'models/best.pt').is_file():shutil.copy2(prev/'models/best.pt',BEST)
    if (prev/'tables/training_history.csv').is_file():shutil.copy2(prev/'tables/training_history.csv',HISTORY)
    print('Resuming from:',resume_path,'at',state['phase'],state['next_epoch'])
model=make_model(load_pretrained=state is None)
if state:
    model.load_state_dict(state['model']);restore_rng(state)
if IMAGENET_WEIGHTS_PATH:
    initial_sha=sha256(IMAGENET_WEIGHTS_PATH)
else:
    from torch.hub import get_dir
    cached=Path(get_dir())/'checkpoints'/'resnet50-0676ba61.pth'
    initial_sha=sha256(cached) if cached.is_file() else None
dump(OUT/'reports/initialization.json',{'initialization':CFG['initialization'],'weight_source':'user_input' if IMAGENET_WEIGHTS_PATH else 'torchvision_hub','weights_sha256':initial_sha})
print('Protocol SHA256:',SIGNATURE,'resume:',bool(state))

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 200MB/s]


Protocol SHA256: 0d8584121ff679d854cdacbfefe4bd50bdda685d4776417ae0090c4e4ed67b22 resume: False


## 4. Train one seed (resumable; leave test untouched)

In [5]:
def train_epoch(model,loader,optimizer,scaler,epoch,phase):
    mode_train(model,phase)
    G.manual_seed(SEED*100000+epoch+(0 if phase=='head' else 10000))
    total=0.;correct=0;n=0
    for x,y in loader:
        x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type,enabled=AMP and DEVICE.type=='cuda'):
            logits=model(x)
            loss=(F.cross_entropy(logits.float(),y,reduction='none')*CLASS_WEIGHTS[y]).mean()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad),1.)
        scaler.step(optimizer);scaler.update()
        correct+=int((logits.argmax(1)==y).sum().item());n+=len(y);total+=float(loss.item())*len(y)
    return total/n,correct/n

history=pd.read_csv(HISTORY).to_dict('records') if HISTORY.exists() else []
phases=[('head',HEAD_EPOCHS,HEAD_LR),('finetune',FINETUNE_EPOCHS,FINETUNE_LR)]
current_phase=state['phase'] if state else 'head'
next_epoch=state['next_epoch'] if state else 0
bad_epochs=state.get('bad_epochs',0) if state else 0
best_key=tuple(state['best_key']) if state and state.get('best_key') is not None else (-1.,-1.,-1.)
status='PAUSED'

for phase,epochs,lr in phases:
    if current_phase=='done':status='COMPLETED';break
    if [p[0] for p in phases].index(phase)<[p[0] for p in phases].index(current_phase):continue
    set_phase(model,phase)
    optimizer=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=lr,weight_decay=WEIGHT_DECAY)
    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='min',factor=.5,patience=3,min_lr=1e-7) if phase=='finetune' else None
    first_epoch=next_epoch if phase==current_phase else 0
    if state and phase==current_phase and state['optimizer'] is not None:
        optimizer.load_state_dict(state['optimizer'])
        if scheduler is not None and state['scheduler'] is not None:scheduler.load_state_dict(state['scheduler'])
    scaler=torch.amp.GradScaler('cuda',enabled=AMP and DEVICE.type=='cuda')
    if state and phase==current_phase and state.get('scaler') is not None:scaler.load_state_dict(state['scaler'])
    if phase=='finetune' and first_epoch==0:
        bad_epochs=0;best_key=(-1.,-1.,-1.)
    print(f'Phase={phase}; epoch={first_epoch+1}/{epochs}; trainable params={sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
    phase_complete=False
    for epoch in range(first_epoch,epochs):
        if time.monotonic()-START>TIME_LIMIT_HOURS*3600:
            print('Runtime guard reached; training state is saved for resumption.')
            break
        t=time.monotonic()
        tr_loss,tr_acc=train_epoch(model,train_loader,optimizer,scaler,epoch,phase)
        val=evaluate(model,val_loader)
        if scheduler:scheduler.step(val['nll'])
        key=(val['balanced_accuracy'],val['macro_f1'],val['accuracy'])
        improved=phase=='finetune' and key>best_key
        if phase=='finetune':
            if improved:
                best_key=key;bad_epochs=0
                tmp=BEST.with_suffix('.tmp')
                torch.save({'model':model.state_dict(),'signature':SIGNATURE,'seed':SEED,'phase':phase,'epoch':epoch+1,'validation_metrics':val},tmp)
                tmp.replace(BEST)
            else:bad_epochs+=1
        row={'seed':SEED,'phase':phase,'epoch':epoch+1,'train_loss':tr_loss,'train_accuracy':tr_acc,
             'val_loss':val['nll'],'val_balanced_accuracy':val['balanced_accuracy'],
             'val_macro_f1':val['macro_f1'],'val_accuracy':val['accuracy'],
             'lr':optimizer.param_groups[0]['lr'],'seconds':time.monotonic()-t,'is_best':int(improved)}
        history.append(row);pd.DataFrame(history).to_csv(HISTORY,index=False)
        save_last(model,phase,epoch+1,optimizer,scheduler,scaler,bad_epochs,list(best_key))
        print(f'{phase} {epoch+1:02d}/{epochs} | train {tr_loss:.4f} | val BACC {val["balanced_accuracy"]:.4f} | F1 {val["macro_f1"]:.4f} | loss {val["nll"]:.4f} | {row["seconds"]:.0f}s'+(' *' if improved else ''))
        if phase=='finetune' and bad_epochs>=PATIENCE:
            print('Early stopping on validation BACC.');phase_complete=True;break
        if epoch==epochs-1:phase_complete=True
    if not phase_complete:break
    if phase=='head':
        current_phase='finetune';next_epoch=0;state=None
        save_last(model,'finetune',0,bad_epochs=0,best_key=[-1.,-1.,-1.])
        continue
    require(BEST.is_file(),'No validation-selected fine-tuning checkpoint was saved')
    current_phase='done';next_epoch=0;status='COMPLETED'
    save_last(model,'done',0,bad_epochs=0,best_key=list(best_key))
if current_phase=='done':status='COMPLETED'
dump(OUT/'reports/training_status.json',{'stage':'R01','status':status,'seed':SEED,'elapsed_seconds':time.monotonic()-START,
    'protocol_sha256':SIGNATURE,'history_rows':len(history),'best_checkpoint_present':BEST.is_file()})
print('TRAINING STATUS:',status)

Phase=head; epoch=1/8; trainable params=16,392
head 01/8 | train 0.3329 | val BACC 0.3331 | F1 0.3291 | loss 1.1503 | 331s
head 02/8 | train 0.2998 | val BACC 0.4019 | F1 0.3811 | loss 1.1531 | 32s
head 03/8 | train 0.2911 | val BACC 0.3927 | F1 0.3869 | loss 1.0526 | 32s
head 04/8 | train 0.2834 | val BACC 0.3983 | F1 0.3995 | loss 1.0605 | 31s
head 05/8 | train 0.2817 | val BACC 0.3900 | F1 0.3893 | loss 0.9914 | 31s
head 06/8 | train 0.2791 | val BACC 0.3847 | F1 0.3540 | loss 1.0730 | 30s
head 07/8 | train 0.2778 | val BACC 0.4003 | F1 0.4135 | loss 1.0206 | 32s
head 08/8 | train 0.2782 | val BACC 0.3889 | F1 0.3772 | loss 1.0426 | 33s
Phase=finetune; epoch=1/35; trainable params=23,471,304
finetune 01/35 | train 0.2624 | val BACC 0.4643 | F1 0.4263 | loss 0.8481 | 73s *
finetune 02/35 | train 0.2129 | val BACC 0.5176 | F1 0.5007 | loss 0.7986 | 72s *
finetune 03/35 | train 0.1830 | val BACC 0.5442 | F1 0.4981 | loss 0.9090 | 72s *
finetune 04/35 | train 0.1591 | val BACC 0.5525 | 

## 5. Export development predictions and journal-ready artifacts

In [6]:
if status=='COMPLETED':
    best=torch.load(BEST,map_location='cpu',weights_only=False)
    require(best['signature']==SIGNATURE,'Best checkpoint protocol mismatch')
    model.load_state_dict(best['model'])
    summaries={}
    for split,loader in [('validation',val_loader),('meta_train',meta_loader)]:
        metrics,y,logits,proba=evaluate(model,loader,keep=True)
        frame=frames[split]
        require(np.array_equal(y,frame.class_index.to_numpy(dtype=np.int64)),f'{split} prediction order mismatch')
        np.savez_compressed(OUT/'predictions'/f'{split}.npz',image_id=frame.image_id.to_numpy(dtype=str),
                            group_id=frame.group_id.to_numpy(dtype=str),y=y.astype('int32'),
                            logits=logits.astype('float32'),proba=proba.astype('float32'),
                            classes=np.array(CLASSES),seed=np.array(SEED),protocol_sha256=np.array(SIGNATURE))
        summaries[split]=metrics
    y=frames['validation'].class_index.to_numpy(dtype=int)
    with np.load(OUT/'predictions/validation.npz',allow_pickle=False) as z:pr=z['proba'].argmax(1)
    cm=confusion_matrix(y,pr,labels=range(len(CLASSES)))
    pd.DataFrame(cm,index=CLASSES,columns=CLASSES).to_csv(OUT/'tables/validation_confusion_matrix.csv')
    pd.DataFrame(classification_report(y,pr,labels=list(range(8)),target_names=CLASSES,output_dict=True,zero_division=0)).T.to_csv(OUT/'tables/validation_classification_report.csv')
    fig,axes=plt.subplots(1,2,figsize=(10,3.5))
    hist=pd.DataFrame(history)
    for phase,sub in hist.groupby('phase',sort=False):
        axes[0].plot(np.arange(len(sub))+1+(0 if phase=='head' else HEAD_EPOCHS),sub.val_balanced_accuracy,label=phase)
        axes[1].plot(np.arange(len(sub))+1+(0 if phase=='head' else HEAD_EPOCHS),sub.train_loss,label=f'{phase} train')
        axes[1].plot(np.arange(len(sub))+1+(0 if phase=='head' else HEAD_EPOCHS),sub.val_loss,label=f'{phase} val')
    axes[0].set(xlabel='Epoch',ylabel='Validation balanced accuracy',title='Supervised ResNet50')
    axes[1].set(xlabel='Epoch',ylabel='Loss',title='Training history')
    for ax in axes:ax.legend(fontsize=8);ax.grid(alpha=.2)
    fig.tight_layout();fig.savefig(OUT/'figures/training_curves.png',dpi=200,bbox_inches='tight');plt.close(fig)
    final={'stage':'R01','status':'COMPLETED','seed':SEED,'initialization':'ImageNet1K V1','best_epoch':best['epoch'],
           'selection_metrics':best['validation_metrics'],'final_validation':summaries['validation'],
           'meta_training_inference_only':True,'training_images':len(frames['train']),
           'validation_images':len(frames['validation']),'meta_images':len(frames['meta_train']),
           'n01_manifest_sha256':protocol['n01_manifest_sha256'],'config_sha256':SIGNATURE,
           'runtime_seconds':time.monotonic()-START,
           'exports':['models/best.pt','training_state/last.pt','tables/training_history.csv','predictions/validation.npz','predictions/meta_train.npz']}
    dump(OUT/'reports/final_metrics.json',final)
    print('SUCCESS — R01 seed',SEED,'validation:',summaries['validation'])
    print('Saved predictions: validation and meta_train only; no test inference.')
else:
    print('PAUSED — attach this output to another Kaggle run of the same seed to resume from training_state/last.pt.')
print('Output root:',OUT)

SUCCESS — R01 seed 42 validation: {'accuracy': 0.7236962488563586, 'balanced_accuracy': 0.6003941518456992, 'macro_f1': 0.5773206243785897, 'nll': 1.011554489851216}
Saved predictions: validation and meta_train only; no test inference.
Output root: /kaggle/working/lesion-r01-supervised-resnet50-s42
